In [1]:
# PyBryt Reference Implementation
# 題目：AI投資分析圖表驗證
# 重點：檢查AI最容易產生幻覺的6個核心位置

import pybryt
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
# 測試資料（模擬真實股價）
np.random.seed(42)
dates = [datetime(2024, 1, 1) + timedelta(days=i) for i in range(60)]
prices = [150.0]
for _ in range(59):
    prices.append(round(prices[-1] * (1 + np.random.normal(0, 0.02)), 2))

# 故意埋入AI容易忽略的edge case
prices[10] = -5.0        # 幻覺點1：負數股價（AI常忘記驗證）
prices[25] = 99999.0     # 幻覺點2：極端離群值（AI常直接畫進去）
prices[40] = None        # 幻覺點3：None 值（AI常假設資料乾淨）

# Reference Implementation（正確解法）
with pybryt.tracing():

    # --- 幻覺點 1：資料清洗（AI 最常跳過這步）---
    # AI 幻覺：直接 plot(dates, prices)，假設資料一定乾淨
    # 正確行為：過濾 None、負數、極端值
    clean_prices = []
    clean_dates = []
    for d, p in zip(dates, prices):
        if p is not None and p > 0 and p < 10000:
            clean_prices.append(p)
            clean_dates.append(d)

    pybryt.Value(
        len(clean_prices),
        name="data_cleaning_removes_invalid",
        msg="[幻覺點1] 資料清洗：應過濾 None、負數、極端值，有效資料筆數應少於原始 60 筆",
        importance=10  # 最高權重：最常被 AI 跳過
    )

    pybryt.Value(
        all(p > 0 for p in clean_prices),
        name="no_negative_prices",
        msg="[幻覺點1] 不應存在負數股價，AI 常假設股價必為正數而不驗證",
        importance=10
    )

    # --- 幻覺點 2：y 軸範圍設定（AI 常產生誤導性圖表）---
    # AI 幻覺：不設定 ylim，讓 matplotlib 自動決定（可能從 0 開始誤導漲跌幅）
    # 正確行為：y 軸應以資料範圍為基準，加適當 padding
    price_min = min(clean_prices)
    price_max = max(clean_prices)
    padding = (price_max - price_min) * 0.1

    expected_ymin = price_min - padding
    expected_ymax = price_max + padding

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(clean_dates, clean_prices, color='#2196F3', linewidth=1.5, label='股價')
    ax.set_ylim(expected_ymin, expected_ymax)

    actual_ymin, actual_ymax = ax.get_ylim()

    pybryt.Value(
        round(actual_ymin, 0),
        name="ylim_min_correct",
        msg="[幻覺點2] y 軸下限應貼近資料最小值（加 padding），AI 常設為 0 導致圖表誤導",
        importance=9
    )

    pybryt.Value(
        actual_ymax > price_max,
        name="ylim_max_has_padding",
        msg="[幻覺點2] y 軸上限應超過最大值（有 padding），否則最高點會被截切",
        importance=9
    )

    # --- 幻覺點 3：日期軸格式（AI 常產生亂碼或重疊）---
    # AI 幻覺：不處理 x 軸日期格式，產生密集重疊的日期文字
    # 正確行為：使用 AutoDateLocator 或 set_major_formatter
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.xticks(rotation=45)

    # 驗證 formatter 類型
    formatter = ax.xaxis.get_major_formatter()
    pybryt.Value(
        isinstance(formatter, mdates.DateFormatter),
        name="date_formatter_applied",
        msg="[幻覺點3] 必須設定 DateFormatter，AI 常忽略日期軸格式化導致標籤重疊",
        importance=8
    )

    # --- 幻覺點 4：移動平均線（AI 常算錯 window 邊界）---
    # AI 幻覺：rolling mean 的前 N-1 筆產生 NaN，AI 常直接畫出或用 0 填補
    # 正確行為：應該 dropna() 或只畫有效部分
    window = 5
    prices_array = np.array(clean_prices)
    ma_full = np.convolve(prices_array, np.ones(window)/window, mode='valid')  # 正確：只有有效值

    pybryt.Value(
        len(ma_full),
        name="moving_average_length_correct",
        msg=f"[幻覺點4] {window}日移動平均長度應為 len(data)-{window-1}，AI 常讓長度與原資料相同（含 NaN）",
        importance=8
    )

    pybryt.Value(
        not any(np.isnan(ma_full)),
        name="moving_average_no_nan",
        msg="[幻覺點4] 移動平均不應包含 NaN，AI 常忽略邊界 NaN 問題",
        importance=8
    )

    # 對應正確的日期（去掉前 window-1 筆）
    ma_dates = clean_dates[window-1:]
    ax.plot(ma_dates, ma_full, color='#FF5722', linewidth=1.5,
            linestyle='--', label=f'{window}日均線')

    # --- 幻覺點 5：圖例與標籤（AI 常忘記或放錯位置）---
    # AI 幻覺：legend() 呼叫但沒有 label 參數，或 legend 遮住重要資料
    ax.set_title('股價走勢圖', fontsize=14, fontweight='bold')
    ax.set_xlabel('日期', fontsize=12)
    ax.set_ylabel('股價 (USD)', fontsize=12)
    legend = ax.legend(loc='best')

    pybryt.Value(
        len(legend.get_texts()) >= 2,
        name="legend_has_entries",
        msg="[幻覺點5] 圖例應包含股價線與均線，AI 常呼叫 legend() 但 label 未設定",
        importance=7
    )

    pybryt.Value(
        ax.get_xlabel() != '',
        name="xlabel_not_empty",
        msg="[幻覺點5] x 軸標籤不應為空，AI 常忘記設定 axis label",
        importance=6
    )

    pybryt.Value(
        ax.get_ylabel() != '',
        name="ylabel_not_empty",
        msg="[幻覺點5] y 軸標籤不應為空，且應標明單位",
        importance=6
    )

    # --- 幻覺點 6：圖表儲存 dpi（AI 常使用預設低解析度）---
    # AI 幻覺：plt.savefig('chart.png') 不設 dpi，預設 72 dpi 模糊
    # 正確行為：商業報告應至少 150 dpi
    fig.savefig('/tmp/test_chart.png', dpi=150, bbox_inches='tight')

    import os
    saved_file_exists = os.path.exists('/tmp/test_chart.png')
    file_size = os.path.getsize('/tmp/test_chart.png') if saved_file_exists else 0

    pybryt.Value(
        saved_file_exists,
        name="chart_saved_successfully",
        msg="[幻覺點6] 圖表應成功儲存為檔案",
        importance=5
    )

    pybryt.Value(
        file_size > 30000,  # 150 dpi 應大於 30KB
        name="chart_dpi_sufficient",
        msg="[幻覺點6] 圖表解析度不足（檔案過小），AI 常忽略 dpi 參數，請設定 dpi >= 150",
        importance=5
    )

    plt.tight_layout()
    plt.close(fig)

AttributeError: module 'pybryt' has no attribute 'tracing'

In [5]:
import pybryt
import numpy as np
import matplotlib

matplotlib.use("Agg")

# 中文字型
matplotlib.rcParams['font.sans-serif'] = ['Microsoft JhengHei']
matplotlib.rcParams['axes.unicode_minus'] = False

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from datetime import datetime, timedelta
import os

# =========================
# 測試資料
# =========================
np.random.seed(42)

dates = [
    datetime(2024, 1, 1) + timedelta(days=i)
    for i in range(60)
]

prices = [150.0]

for _ in range(59):
    prices.append(
        round(
            prices[-1] * (1 + np.random.normal(0, 0.02)),
            2
        )
    )

# Edge cases
prices[10] = -5.0
prices[25] = 99999.0
prices[40] = None

# =========================
# 幻覺點1：資料清洗
# =========================
clean_dates = []
clean_prices = []

for d, p in zip(dates, prices):

    if (
        p is not None
        and p > 0
        and p < 10000
    ):
        clean_dates.append(d)
        clean_prices.append(p)

ann1 = pybryt.Value(
    len(clean_prices),
    name="data_cleaning_removes_invalid"
)

ann2 = pybryt.Value(
    all(p > 0 for p in clean_prices),
    name="no_negative_prices"
)

ann3 = pybryt.Value(
    all(p < 10000 for p in clean_prices),
    name="remove_outliers"
)

# =========================
# 幻覺點2：Y 軸設定
# =========================
price_min = min(clean_prices)
price_max = max(clean_prices)

padding = (price_max - price_min) * 0.1

expected_ymin = price_min - padding
expected_ymax = price_max + padding

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(
    clean_dates,
    clean_prices,
    label="股價"
)

ax.set_ylim(
    expected_ymin,
    expected_ymax
)

actual_ymin, actual_ymax = ax.get_ylim()

ann4 = pybryt.Value(
    round(actual_ymin, 0),
    name="ylim_min_correct"
)

ann5 = pybryt.Value(
    actual_ymax > price_max,
    name="ylim_max_has_padding"
)

# =========================
# 幻覺點3：日期格式
# =========================
ax.xaxis.set_major_formatter(
    mdates.DateFormatter("%Y-%m")
)

ax.xaxis.set_major_locator(
    mdates.MonthLocator()
)

plt.xticks(rotation=45)

formatter = ax.xaxis.get_major_formatter()

ann6 = pybryt.Value(
    isinstance(formatter, mdates.DateFormatter),
    name="date_formatter_applied"
)

# =========================
# 幻覺點4：移動平均
# =========================
window = 5

prices_array = np.array(clean_prices)

ma_full = np.convolve(
    prices_array,
    np.ones(window) / window,
    mode="valid"
)

ann7 = pybryt.Value(
    len(ma_full),
    name="moving_average_length_correct"
)

ann8 = pybryt.Value(
    not np.isnan(ma_full).any(),
    name="moving_average_no_nan"
)

ma_dates = clean_dates[window - 1:]

ax.plot(
    ma_dates,
    ma_full,
    label=f"{window}日均線"
)

# =========================
# 幻覺點5：圖例與標籤
# =========================
ax.set_title("股價走勢圖")

ax.set_xlabel("日期")
ax.set_ylabel("股價 (USD)")

legend = ax.legend(loc="best")

ann9 = pybryt.Value(
    len(legend.get_texts()) >= 2,
    name="legend_has_entries"
)

ann10 = pybryt.Value(
    ax.get_xlabel() != "",
    name="xlabel_not_empty"
)

ann11 = pybryt.Value(
    ax.get_ylabel() != "",
    name="ylabel_not_empty"
)

# =========================
# 幻覺點6：圖表儲存
# =========================
save_dir = "tmp"

os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(
    save_dir,
    "test_chart.png"
)

fig.savefig(
    save_path,
    dpi=150,
    bbox_inches="tight"
)

saved_file_exists = os.path.exists(save_path)

file_size = (
    os.path.getsize(save_path)
    if saved_file_exists
    else 0
)

ann12 = pybryt.Value(
    saved_file_exists,
    name="chart_saved_successfully"
)

ann13 = pybryt.Value(
    file_size > 30000,
    name="chart_dpi_sufficient"
)

plt.close(fig)

# =========================
# 建立 Reference
# =========================
ref = pybryt.ReferenceImplementation(
    "ai_investment_chart_reference",
    [
        ann1,
        ann2,
        ann3,
        ann4,
        ann5,
        ann6,
        ann7,
        ann8,
        ann9,
        ann10,
        ann11,
        ann12,
        ann13
    ]
)

# 匯出
ref.dump("ai_investment_chart_reference.pkl")

print("Reference 建立成功")

Reference 建立成功


In [7]:
#正確程式碼
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
dates = [datetime(2024,1,1)+timedelta(days=i) for i in range(60)]
prices = [150.0]
for _ in range(59):
    prices.append(round(prices[-1]*(1+np.random.normal(0,0.02)),2))

# 故意埋入的問題資料
prices[10] = -5.0
prices[25] = 99999.0
prices[40] = None

# 1. 資料清洗：過濾 None、負數、極端值
clean_prices, clean_dates = [], []
for d, p in zip(dates, prices):
    if p is not None and p > 0 and p < 10000:
        clean_prices.append(p)
        clean_dates.append(d)

fig, ax = plt.subplots(figsize=(12,6))

# 2. y 軸範圍：以資料為基準加 padding
price_min, price_max = min(clean_prices), max(clean_prices)
padding = (price_max - price_min) * 0.1
ax.set_ylim(price_min - padding, price_max + padding)

ax.plot(clean_dates, clean_prices, color='#2196F3', linewidth=1.5, label='股價')

# 3. 日期軸格式化
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator())
plt.xticks(rotation=45)

# 4. 移動平均（無 NaN 版本）
window = 5
prices_arr = np.array(clean_prices)
ma = np.convolve(prices_arr, np.ones(window)/window, mode='valid')
ma_dates = clean_dates[window-1:]
ax.plot(ma_dates, ma, color='#FF5722', linestyle='--', linewidth=1.5, label='5日均線')

# 5. 圖例與標籤
ax.set_xlabel('日期', fontsize=12)
ax.set_ylabel('股價 (USD)', fontsize=12)
ax.set_title('股價走勢圖')
ax.legend(loc='best')

# 6. 高解析度儲存
fig.savefig('chart.png', dpi=150, bbox_inches='tight')
plt.tight_layout()

In [2]:
# ============================================
# PyBryt Student Reference
# AI 投資分析圖表驗證
# ============================================

import pybryt
import numpy as np
import matplotlib

# 使用非 GUI backend
matplotlib.use("Agg")

# =========================
# 中文字型修正
# =========================
# Windows 常用：
matplotlib.rcParams['font.sans-serif'] = ['Microsoft JhengHei']

# 避免負號亂碼
matplotlib.rcParams['axes.unicode_minus'] = False

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from datetime import datetime, timedelta
import os

# ============================================
# 建立測試資料
# ============================================
np.random.seed(42)

dates = [
    datetime(2024, 1, 1) + timedelta(days=i)
    for i in range(60)
]

prices = [150.0]

for _ in range(59):

    prices.append(
        round(
            prices[-1] * (
                1 + np.random.normal(0, 0.02)
            ),
            2
        )
    )

# ============================================
# 故意埋入問題資料
# ============================================
prices[10] = -5.0
prices[25] = 99999.0
prices[40] = None

# ============================================
# 1. 資料清洗
# ============================================
clean_prices = []
clean_dates = []

for d, p in zip(dates, prices):

    if (
        p is not None
        and p > 0
        and p < 10000
    ):

        clean_prices.append(p)
        clean_dates.append(d)

# PyBryt annotations
ann1 = pybryt.Value(
    len(clean_prices),
    name="clean_data_length"
)

ann2 = pybryt.Value(
    all(p > 0 for p in clean_prices),
    name="no_negative_values"
)

ann3 = pybryt.Value(
    all(p < 10000 for p in clean_prices),
    name="outlier_removed"
)

# ============================================
# 建立圖表
# ============================================
fig, ax = plt.subplots(figsize=(12, 6))

# ============================================
# 2. Y 軸範圍設定
# ============================================
price_min = min(clean_prices)
price_max = max(clean_prices)

padding = (
    price_max - price_min
) * 0.1

ax.set_ylim(
    price_min - padding,
    price_max + padding
)

actual_ylim = ax.get_ylim()

ann4 = pybryt.Value(
    actual_ylim[0],
    name="ymin_set"
)

ann5 = pybryt.Value(
    actual_ylim[1],
    name="ymax_set"
)

# ============================================
# 股價線
# ============================================
ax.plot(
    clean_dates,
    clean_prices,
    color='#2196F3',
    linewidth=1.5,
    label='股價'
)

# ============================================
# 3. 日期格式化
# ============================================
ax.xaxis.set_major_formatter(
    mdates.DateFormatter('%Y-%m')
)

ax.xaxis.set_major_locator(
    mdates.MonthLocator()
)

plt.xticks(rotation=45)

formatter = ax.xaxis.get_major_formatter()

ann6 = pybryt.Value(
    isinstance(
        formatter,
        mdates.DateFormatter
    ),
    name="date_formatter_used"
)

# ============================================
# 4. 移動平均
# ============================================
window = 5

prices_arr = np.array(clean_prices)

ma = np.convolve(
    prices_arr,
    np.ones(window) / window,
    mode='valid'
)

ma_dates = clean_dates[
    window - 1:
]

ann7 = pybryt.Value(
    len(ma),
    name="moving_average_length"
)

ann8 = pybryt.Value(
    not np.isnan(ma).any(),
    name="moving_average_no_nan"
)

# ============================================
# 均線
# ============================================
ax.plot(
    ma_dates,
    ma,
    color='#FF5722',
    linestyle='--',
    linewidth=1.5,
    label='5日均線'
)

# ============================================
# 5. 圖例與標籤
# ============================================
ax.set_xlabel(
    '日期',
    fontsize=12
)

ax.set_ylabel(
    '股價 (USD)',
    fontsize=12
)

ax.set_title(
    '股價走勢圖'
)

legend = ax.legend(
    loc='best'
)

ann9 = pybryt.Value(
    len(legend.get_texts()) >= 2,
    name="legend_exists"
)

ann10 = pybryt.Value(
    ax.get_xlabel() != '',
    name="xlabel_exists"
)

ann11 = pybryt.Value(
    ax.get_ylabel() != '',
    name="ylabel_exists"
)

# ============================================
# 6. 高解析度儲存
# ============================================
save_path = "chart.png"

fig.savefig(
    save_path,
    dpi=150,
    bbox_inches='tight'
)

saved = os.path.exists(
    save_path
)

file_size = (
    os.path.getsize(save_path)
    if saved
    else 0
)

ann12 = pybryt.Value(
    saved,
    name="chart_saved"
)

ann13 = pybryt.Value(
    file_size > 30000,
    name="high_resolution_chart"
)

# ============================================
# Layout 修正
# ============================================
plt.tight_layout()

# 關閉圖表
plt.close(fig)

# ============================================
# 建立 Student Reference
# ============================================
student_reference = pybryt.ReferenceImplementation(
    "student_reference",
    [
        ann1,
        ann2,
        ann3,
        ann4,
        ann5,
        ann6,
        ann7,
        ann8,
        ann9,
        ann10,
        ann11,
        ann12,
        ann13
    ]
)

# ============================================
# 匯出 pkl
# ============================================
student_reference.dump(
    "student_reference.pkl"
)

# ============================================
# 完成訊息
# ============================================
print("student_reference.pkl 建立成功")
print(f"圖表已儲存：{save_path}")
print(f"檔案大小：{file_size} bytes")

student_reference.pkl 建立成功
圖表已儲存：chart.png
檔案大小：78710 bytes


In [8]:
#AI幻覺
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)
dates = [datetime(2024,1,1)+timedelta(days=i) for i in range(60)]
prices = [150.0]
for _ in range(59):
    prices.append(round(prices[-1]*(1+np.random.normal(0,0.02)),2))

# AI 直接使用原始資料（未清洗）
prices[10] = -5.0
prices[25] = 99999.0
prices[40] = None

fig, ax = plt.subplots(figsize=(12,6))

# AI 幻覺：從 0 開始的 y 軸
ax.set_ylim(0, max(p for p in prices if p))

# AI 幻覺：直接 plot 所有資料（含 None）
valid = [p if p else 0 for p in prices]
ax.plot(dates, valid)

# AI 幻覺：rolling mean 含 NaN
import pandas as pd
s = pd.Series(prices)
ma = s.rolling(5).mean().fillna(0)
ax.plot(dates, ma)

# AI 幻覺：沒有 DateFormatter、沒有 label、沒有 dpi
ax.legend()
plt.savefig('chart.png')

No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


In [6]:
import pybryt

# =========================
# 1. 載入 reference
# =========================
reference = pybryt.ReferenceImplementation.load(
    "student_reference.pkl"
)

# =========================
# 2. 讀取 student.py（關鍵）
# =========================
with open("student_submission.py", "r", encoding="utf-8") as f:
    student_code = f.read()

# =========================
# 3. 建立 StudentImplementation（舊版正確方式）
# =========================
student_impl = pybryt.StudentImplementation(
    student_code
)

# =========================
# 4. 比對
# =========================
result = reference.run(student_impl)

# =========================
# 5. 輸出結果
# =========================
print("是否通過：", result.correct)

for r in result.results:
    print(r.annotation.name, "=>", r.satisfied)

FileNotFoundError: [Errno 2] No such file or directory: 'student_submission.py'